# 03 – So sánh các thuật toán học máy và chọn mô hình tốt nhất

Notebook này chạy **6 thuật toán học máy** trên cùng một bộ đặc trưng và cùng một cách chia dữ liệu, để so sánh công bằng:

| Tên | Thuật toán | Ý tưởng chính |
|---|---|---|
| E1 | Logistic Regression | Đường phân tách tuyến tính + hàm sigmoid |
| E2 | K-Nearest Neighbors (k=5) | Gán nhãn theo đa số của 5 ảnh gần nhất |
| E3 | Gaussian Naive Bayes | Định lý Bayes, giả định các đặc trưng độc lập |
| E4 | Decision Tree (depth 10) | Chuỗi câu hỏi if/else trên đặc trưng |
| E5 | Random Forest (200 cây) | Nhiều cây quyết định bỏ phiếu (ensemble, bagging) |
| E6 | SVM kernel RBF (C=10) | Siêu phẳng lề lớn nhất trong không gian kernel |

**Quy tắc chọn:** `val_acc` cao nhất (hòa thì `val_loss` thấp hơn), chỉ dùng tập validation. Tập test chỉ dùng một lần cho mô hình thắng.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import json
import matplotlib.pyplot as plt
from IPython.display import Image

from src.data_loader import BEST_CONFIG_PATH, REPORTS_DIR
from src.experiments import run_experiments

## 1. Chạy các thí nghiệm

Mô hình thắng sẽ được đánh giá trên tập test và lưu vào `models/best_model.joblib` (API sẽ dùng file này). SVM và KNN có thể chậm hơn các thuật toán còn lại.

In [ ]:
results, info = run_experiments()
results[['name', 'val_acc', 'val_f1', 'val_loss', 'seconds']]

In [ ]:
Image(os.path.join(REPORTS_DIR, 'experiments_comparison.png'))

## 2. Kết quả chốt

Cấu hình thắng, điểm validation và điểm test cuối cùng.

In [ ]:
print(json.dumps(info, ensure_ascii=False, indent=2))
Image(os.path.join(REPORTS_DIR, 'confusion_matrix.png'))

## 3. Chọn siêu tham số: ảnh hưởng của `k` trong KNN

**Bias–variance:** `k` nhỏ thì mô hình rất linh hoạt (dễ overfit, accuracy train cao nhưng val thấp); `k` lớn thì mô hình mượt hơn nhưng có thể quá đơn giản (underfit). Điểm tốt nhất thường nằm ở giữa.

In [ ]:
from sklearn.metrics import accuracy_score
from src.features import load_feature_splits
from src.train import build_model

data = load_feature_splits()
X_train, y_train = data['train']
X_val, y_val = data['val']

ks = [1, 3, 5, 7, 11, 15, 21, 31]
tr_acc, va_acc = [], []
for k in ks:
    m = build_model({'algo': 'knn', 'params': {'n_neighbors': k}}).fit(X_train, y_train)
    tr_acc.append(accuracy_score(y_train, m.predict(X_train)))
    va_acc.append(accuracy_score(y_val, m.predict(X_val)))

plt.plot(ks, tr_acc, marker='o', label='Train')
plt.plot(ks, va_acc, marker='o', label='Validation')
plt.xlabel('k (số láng giềng)'); plt.ylabel('Accuracy'); plt.title('KNN: ảnh hưởng của k')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 4. Overfitting với Decision Tree: ảnh hưởng của `max_depth`

Cây càng sâu càng khớp chặt dữ liệu train. Khi khoảng cách giữa đường Train và Validation lớn dần là dấu hiệu **overfitting**.

In [ ]:
depths = [1, 2, 3, 5, 8, 12, 20, None]
tr_acc, va_acc = [], []
for d in depths:
    m = build_model({'algo': 'decision_tree', 'params': {'max_depth': d}}).fit(X_train, y_train)
    tr_acc.append(accuracy_score(y_train, m.predict(X_train)))
    va_acc.append(accuracy_score(y_val, m.predict(X_val)))

labels = [str(d) for d in depths]
plt.plot(labels, tr_acc, marker='o', label='Train')
plt.plot(labels, va_acc, marker='o', label='Validation')
plt.xlabel('max_depth (None = không giới hạn)'); plt.ylabel('Accuracy'); plt.title('Decision Tree: ảnh hưởng của độ sâu')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5. Độ quan trọng của đặc trưng

Nếu mô hình thắng là Random Forest hoặc Decision Tree, biểu đồ này cho biết đặc trưng nào đóng góp nhiều nhất vào việc phân biệt `fresh` và `rotten`.

In [ ]:
path = os.path.join(REPORTS_DIR, 'feature_importance.png')
Image(path) if os.path.exists(path) else print('Mô hình thắng không có feature_importances_ (không phải thuật toán dựa trên cây).')